In [3]:
import subprocess, os

def run(cmd):
    print(f">>> {cmd[:70]}")
    result = subprocess.run(cmd, shell=True)
    # Status icons for success (✅) or failure (❌)
    print(f"{'✅' if result.returncode == 0 else '❌'} rc={result.returncode}")
    return result.returncode

# Check GPU availability
result = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader",
                        shell=True, capture_output=True, text=True)
gpu = result.stdout.strip()

if not gpu:
    raise SystemExit("⚠️ No GPU detected! Please go to Runtime → Change runtime type → T4 GPU")

print(f"✅ GPU: {gpu}")

# Install packages
print("\n📦 Installing packages...")
# Uninstall existing versions to avoid conflicts
run("pip uninstall -y marker-pdf surya-ocr pdftext 2>/dev/null || true")
run("pip install -q 'marker-pdf==1.10.2'")
run("pip install -q 'protobuf==3.20.3' --force-reinstall")

print("\n✅ Installation complete. Proceed to Cell 3.")

✅ GPU: Tesla T4

📦 Installing packages...
>>> pip uninstall -y marker-pdf surya-ocr pdftext 2>/dev/null || true
✅ rc=0
>>> pip install -q 'marker-pdf==1.10.2'
✅ rc=0
>>> pip install -q 'protobuf==3.20.3' --force-reinstall
✅ rc=0

✅ Installation complete. Proceed to Cell 3.


In [4]:
import torch
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

print(f"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise SystemExit("⚠️ CUDA is unavailable. Please make sure to switch to a GPU runtime.")

from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered
print("✅ Marker imports OK")

print("\n⏳ Loading models into GPU... (Initial setup: ~2-3 mins)")
artifact_dict = create_model_dict()
print(f"✅ Models loaded: {list(artifact_dict.keys())}")

converter = PdfConverter(artifact_dict=artifact_dict)
print("✅ PdfConverter ready — proceed to Cell 4")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Torch: 2.10.0+cu128, CUDA: True
✅ Marker imports OK

⏳ Loading models into GPU... (Initial setup: ~2-3 mins)


✅ Models loaded: ['layout_model', 'recognition_model', 'table_rec_model', 'detection_model', 'ocr_error_model']
✅ PdfConverter ready — proceed to Cell 4


In [5]:
from pathlib import Path

# Set output directory path
OUTPUT_DIR = Path("/content/drive/MyDrive/Uchicago/Capstone/1/Assignment3/parsed_output")

# Filter directories within the output path
dirs = [p for p in OUTPUT_DIR.iterdir() if p.is_dir()]

# Check for the existence of a markdown file matching the folder name (using p.name instead of p.stem)
has_md = [p for p in dirs if (p / f"{p.name}.md").exists()]
no_md  = [p for p in dirs if not (p / f"{p.name}.md").exists()]

print(f"Total directories      : {len(dirs)}")
print(f"Folders with .md files : {len(has_md)}")
print(f"Folders without .md    : {len(no_md)}")

if no_md:
    print(f"\nMissing .md file list: {[p.name for p in no_md]}")

Total directories      : 75
Folders with .md files : 75
Folders without .md    : 0


In [ ]:
from pathlib import Path
import re

OUTPUT_DIR = Path("/content/drive/MyDrive/Uchicago/Capstone/1/Assignment3/parsed_output")
PDF_DIR    = Path("/content/drive/MyDrive/Uchicago/Capstone/1/Assignment3/pdfs")

# Identify a successfully processed file
done = [p for p in OUTPUT_DIR.iterdir() if p.is_dir() and (p / f"{p.name}.md").exists()]
test_id = done[0].name
test_pdf = PDF_DIR / f"{test_id}.pdf"
print(f"Testing target: {test_id}")
print(f"PDF exists: {test_pdf.exists()}")

# Execute conversion (In-memory, output check only)
rendered = converter(str(test_pdf))
text, _, images = text_from_rendered(rendered)

print(f"\n✅ Conversion successful")
print(f"Character count: {len(text)}")
print(f"Image count: {len(images)}")

# Test image path correction logic (Regex)
text_fixed = re.sub(r'!\[(.*?)\]\(((?!images/).*?)\)', r'![\1](images/\2)', text)
img_tags_before = len(re.findall(r'!\[', text))
img_tags_after  = len(re.findall(r'!\[', text_fixed))
print(f"Image placeholders (original): {img_tags_before}")
print(f"Image placeholders (fixed): {img_tags_after}")

# Compare with the original markdown file
original_md = (OUTPUT_DIR / test_id / f"{test_id}.md").read_text(encoding="utf-8")
print(f"\nOriginal .md char count: {len(original_md)}")
print(f"New conversion char count: {len(text_fixed)}")
print(f"Difference: {abs(len(original_md) - len(text_fixed))} characters")

# Preview content without saving (First 300 characters)
print(f"\n--- New Conversion Preview (First 300 chars) ---")
print(text_fixed[:300])

測試對象: 2406.15888v2
PDF 存在: True


Recognizing Text: 100%|██████████| 374/374 [00:11<00:00, 33.14it/s]



✅ 轉換成功
字元數: 48122
圖片數: 1
圖片 placeholder 數: 1
修正後 placeholder 數: 1

原始 md 字元數: 48135
新轉換字元數  : 48129
差異          : 6 字元

--- 新轉換前 300 字 ---
# **Real-time Speech Summarization for Medical Conversations**

Khai Le-Duc\*1,2,6, Khai-Nguyen Nguyen\*3, Long Vo-Dang<sup>4</sup>, Truong-Son Hy<sup>5,6</sup>

<sup>1</sup>University of Toronto, Canada
 <sup>2</sup>University Health Network, Canada
 <sup>3</sup>College of William and Mary, United 


In [ ]:
import json
import re
from pathlib import Path
from tqdm import tqdm

PDF_DIR    = Path("/content/drive/MyDrive/Uchicago/Capstone/1/Assignment3/pdfs")
OUTPUT_DIR = Path("/content/drive/MyDrive/Uchicago/Capstone/1/Assignment3/parsed_output")
OUTPUT_DIR.mkdir(exist_ok=True)

pdf_files = list(PDF_DIR.glob("*.pdf"))

# Fix: Use p.name instead of p.stem to match directory structure
already_done = [
    p.name for p in OUTPUT_DIR.iterdir()
    if p.is_dir() and (p / f"{p.name}.md").exists()
]
print(f"📄 Total PDFs   : {len(pdf_files)}")
print(f"⏭️  Already done : {len(already_done)}")
print(f"🔄 Remaining    : {len(pdf_files) - len(already_done)}")

results = {}
failed  = []

for pdf_path in tqdm(pdf_files, desc="Processing PDFs"):
    arxiv_id = pdf_path.stem

    doc_dir = OUTPUT_DIR / arxiv_id
    doc_dir.mkdir(exist_ok=True)
    img_dir = doc_dir / "images"
    img_dir.mkdir(exist_ok=True)

    out_path = doc_dir / f"{arxiv_id}.md"

    # Checkpoint: Resume from where it left off
    if out_path.exists():
        results[arxiv_id] = {"status": "skipped"}
        continue

    try:
        rendered = converter(str(pdf_path))
        text, _, images = text_from_rendered(rendered)

        # Save extracted images
        image_list = []
        for img_name, img_obj in images.items():
            img_obj.save(img_dir / img_name)
            image_list

📄 Total PDFs   : 75
⏭️  Already done : 57
🔄 Remaining    : 18


Recognizing Layout: 100%|██████████| 21/21 [00:14<00:00,  1.49it/s]

Running OCR Error Detection: 100%|██████████| 2/2 [00:00<00:00, 18.09it/s]

Detecting bboxes: 100%|██████████| 2/2 [00:07<00:00,  3.94s/it]

Recognizing Text: 100%|██████████| 312/312 [01:51<00:00,  2.80it/s]

Recognizing Text: 100%|██████████| 13/13 [00:07<00:00,  1.63it/s]

Recognizing tables: 100%|██████████| 1/1 [00:01<00:00,  1.18s/it]

Detecting bboxes: 100%|██████████| 1/1 [00:02<00:00,  2.40s/it]

Processing PDFs:   1%|▏         | 1/75 [03:11<3:55:48, 191.19s/it]

❌ Failed: 2412.00651v1 — TypeError: function takes at most 16 arguments (17 given)



Recognizing Layout: 100%|██████████| 18/18 [00:10<00:00,  1.66it/s]

Running OCR Error Detection: 100%|██████████| 2/2 [00:00<00:00, 27.21it/s]

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  1.19it/s]

Recognizing Text: 100%|██████████| 56/56 [03:24<00:00,  3.65s/it]

Recognizing Text: 100%|██████████| 7/7 [00:03<00:00,  2.31it/s]

Recognizing tables: 100%|██████████| 1/1 [00:00<00:00,  3.80it/s]

Detecting bboxes: 0it [00:00, ?it/s]
Processing PDFs:   4%|▍         | 3/75 [07:07<2:44:42, 137.26s/it]

❌ Failed: 2411.18627v2 — TypeError: function takes at most 16 arguments (17 given)



Processing PDFs:   8%|▊         | 6/75 [07:30<1:26:24, 75.14s/it] 


KeyboardInterrupt: 

In [ ]:
import json
import re
from pathlib import Path
from tqdm import tqdm

PDF_DIR    = Path("/content/drive/MyDrive/Uchicago/Capstone/1/Assignment3/pdfs")
OUTPUT_DIR = Path("/content/drive/MyDrive/Uchicago/Capstone/1/Assignment3/parsed_output")

# Target specific failed IDs for reprocessing
failed_ids = [
    '2410.10516v3', '2412.00651v1', '2411.18627v2', '2412.07587v6',
    '2411.04376v2', '2410.18929v2', '2410.20812v3', '2412.07042v1',
    '2410.13267v2', '2410.07168v2', '2411.11853v3', '2501.01454v2',
    '2412.12783v2', '2411.00816v2', '2411.08777v2', '2410.08642v2',
    '2411.04950v4', '2411.05000v2'
]

print(f"Target count: {len(failed_ids)} files")

results = {}
failed  = []

for arxiv_id in tqdm(failed_ids, desc="Processing failed PDFs"):
    pdf_path = PDF_DIR / f"{arxiv_id}.pdf"

    if not pdf_path.exists():
        print(f"⚠️  PDF not found: {arxiv_id}")
        failed.append(arxiv_id)
        continue

    doc_dir = OUTPUT_DIR / arxiv_id
    doc_dir.mkdir(exist_ok=True)
    img_dir = doc_dir / "images"
    img_dir.mkdir(exist_ok=True)

    out_path = doc_dir / f"{arxiv_id}.md"

    # Checkpoint skip logic
    if out_path.exists():
        print(f"⏭️  Skip: {arxiv_id}")
        results[arxiv_id] = {"status": "skipped"}
        continue

    try:
        rendered = converter(str(pdf_path))
        text, _, images = text_from_rendered(rendered)

        # Process and save images
        image_list = []
        for img_name, img_obj in images.items():
            img_obj.save(img_dir / img_name)
            image_list.append(img_name)

        # Adjust image paths for Markdown
        text = re.sub(r'!\[(.*?)\]\((.*?)\)', r'![\1](images/\2)', text)

        # Append gallery if no inline images are detected
        if images and "![" not in text:
            text += "\n\n## Visual Appendix (Extracted Figures)\n"
            for img_name in image_list:
                text += f"![{img_name}](images/{img_name})\n"

        out_path.write_text(text, encoding="utf-8")
        results[arxiv_id] = {"chars": len(text), "images_count": len(images), "status": "success"}
        print(f"✅ {arxiv_id}: {len(text)} chars, {len(images)} images")

    except Exception as e:
        print(f"❌ {arxiv_id}: {type(e).__name__}: {e}")
        failed.append(arxiv_id)

# Final Status Update
print(f"\n✅ Success : {len([v for v in results.values() if v.get('status') == 'success'])}")
print(f"⏭️  Skipped : {len([v for v in results.values() if v.get('status') == 'skipped'])}")
print(f"❌ Failed  : {len(failed)}")
if failed:
    print(f"Failed IDs : {failed}")

目標: 18 份


Recognizing Layout: 100%|██████████| 25/25 [00:14<00:00,  1.71it/s]

Running OCR Error Detection: 100%|██████████| 2/2 [00:00<00:00, 18.38it/s]

Detecting bboxes: 100%|██████████| 2/2 [00:05<00:00,  2.91s/it]

Recognizing Text: 100%|██████████| 149/149 [02:01<00:00,  1.23it/s]

Recognizing Text: 100%|██████████| 32/32 [03:05<00:00,  5.79s/it]

Recognizing tables: 100%|██████████| 1/1 [00:00<00:00,  1.05it/s]

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  2.41it/s]

Processing failed PDFs:   6%|▌         | 1/18 [05:55<1:40:48, 355.80s/it]

❌ 2410.10516v3: TypeError: function takes at most 16 arguments (17 given)



Recognizing Layout: 100%|██████████| 21/21 [00:13<00:00,  1.61it/s]

Running OCR Error Detection: 100%|██████████| 2/2 [00:00<00:00, 21.23it/s]

Detecting bboxes: 100%|██████████| 2/2 [00:01<00:00,  1.01it/s]

Recognizing Text: 100%|██████████| 312/312 [01:52<00:00,  2.77it/s]

Recognizing Text: 100%|██████████| 13/13 [00:07<00:00,  1.63it/s]

Recognizing tables: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it]

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  1.28it/s]

Processing failed PDFs:  11%|█         | 2/18 [08:54<1:07:09, 251.86s/it]

❌ 2412.00651v1: TypeError: function takes at most 16 arguments (17 given)


...

❌ 2410.08642v2: TypeError: function takes at most 16 arguments (17 given)



Recognizing Layout: 100%|██████████| 79/79 [00:47<00:00,  1.68it/s]

Running OCR Error Detection: 100%|██████████| 6/6 [00:00<00:00, 15.98it/s]

Detecting bboxes: 100%|██████████| 2/2 [00:05<00:00,  2.57s/it]

Recognizing Text: 100%|██████████| 52/52 [01:20<00:00,  1.56s/it]

Recognizing Text: 100%|██████████| 7/7 [00:02<00:00,  2.85it/s]

Recognizing tables: 100%|██████████| 1/1 [00:01<00:00,  1.73s/it]

Detecting bboxes: 0it [00:00, ?it/s]
Processing failed PDFs:  94%|█████████▍| 17/18 [49:39<02:21, 141.38s/it]

❌ 2411.04950v4: TypeError: function takes at most 16 arguments (17 given)



Recognizing Layout: 100%|██████████| 22/22 [00:13<00:00,  1.68it/s]

Running OCR Error Detection: 100%|██████████| 2/2 [00:00<00:00, 20.40it/s]

Detecting bboxes: 100%|██████████| 2/2 [00:02<00:00,  1.15s/it]

Recognizing Text: 100%|██████████| 198/198 [02:40<00:00,  1.24it/s]

Recognizing tables: 100%|██████████| 1/1 [00:01<00:00,  1.58s/it]

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  1.01it/s]

Processing failed PDFs: 100%|██████████| 18/18 [53:40<00:00, 178.93s/it]

❌ 2411.05000v2: TypeError: function takes at most 16 arguments (17 given)

✅ Success : 0
⏭️  Skipped : 0
❌ Failed  : 18
Failed IDs : ['2410.10516v3', '2412.00651v1', '2411.18627v2', '2412.07587v6', '2411.04376v2', '2410.18929v2', '2410.20812v3', '2412.07042v1', '2410.13267v2', '2410.07168v2', '2411.11853v3', '2501.01454v2', '2412.12783v2', '2411.00816v2', '2411.08777v2', '2410.08642v2', '2411.04950v4', '2411.05000v2']


In [ ]:
import subprocess
subprocess.run("pip install -q pymupdf", shell=True)

import fitz  # pymupdf
import json
from pathlib import Path
from tqdm import tqdm

PDF_DIR    = Path("/content/drive/MyDrive/Uchicago/Capstone/1/Assignment3/pdfs")
OUTPUT_DIR = Path("/content/drive/MyDrive/Uchicago/Capstone/1/Assignment3/parsed_output_pymu")
OUTPUT_DIR.mkdir(exist_ok=True)

# Processing the 18 specific failed IDs using PyMuPDF as a fallback
failed_ids = [
    '2410.10516v3', '2412.00651v1', '2411.18627v2', '2412.07587v6',
    '2411.04376v2', '2410.18929v2', '2410.20812v3', '2412.07042v1',
    '2410.13267v2', '2410.07168v2', '2411.11853v3', '2501.01454v2',
    '2412.12783v2', '2411.00816v2', '2411.08777v2', '2410.08642v2',
    '2411.04950v4', '2411.05000v2'
]

print(f"Target count: {len(failed_ids)} files")

results = {}
failed  = []

for arxiv_id in tqdm(failed_ids, desc="Processing with PyMuPDF"):
    pdf_path = PDF_DIR / f"{arxiv_id}.pdf"

    if not pdf_path.exists():
        print(f"⚠️  PDF not found: {arxiv_id}")
        failed.append(arxiv_id)
        continue

    doc_dir = OUTPUT_DIR / arxiv_id
    doc_dir.mkdir(exist_ok=True)
    img_dir = doc_dir / "images"
    img_dir.mkdir(exist_ok=True)

    out_path = doc_dir / f"{arxiv_id}.md"

    # Checkpoint skip logic
    if out_path.exists():
        print(f"⏭️  Skip: {arxiv_id}")
        results[arxiv_id] = {"status": "skipped"}
        continue

    try:
        doc = fitz.open(str(pdf_path))
        pages_text = []
        image_count = 0

        for page_num, page in enumerate(doc):
            # Extract text
            text = page.get_text("text").strip()
            if text:
                pages_text.append(f"\n{text}")

            # Extract images
            for img_index, img in enumerate(page.get_images(full=True)):
                xref = img[0]
                try:
                    base_image = doc.extract_image(xref)
                    img_bytes = base_image["image"]
                    img_ext   = base_image["ext"]
                    img_name  = f"page_{page_num + 1}_img_{img_index}.{img_ext}"
                    (img_dir / img_name).write_bytes(img_bytes)
                    image_count += 1
                except Exception as img_e:
                    print(f"  ⚠️  Image extraction failed for {arxiv_id} p{page_num+1} img{img_index}: {img_e}")

        doc.close()

        # Compose markdown content
        full_text = "\n\n".join(pages_text)

        out_path.write_text(full_text, encoding="utf-8")
        results[arxiv_id] = {
            "chars":       len(full_text),
            "pages":       len(pages_text),
            "images":      image_count,
            "status":      "success"
        }
        print(f"✅ {arxiv_id}: {len(full_text)} chars, {len(pages_text)} pages, {image_count} images")

    except Exception as e:
        print(f"❌ {arxiv_id}: {type(e).__name__}: {e}")
        failed.append(arxiv_id)

# Generate Summary Report
report = {
    "total":   len(failed_ids),
    "success": len([v for v in results.values() if v.get("status") == "success"]),
    "skipped": len([v for v in results.values() if v.get("status") == "skipped"]),
    "failed":  failed,
    "details": {k: v for k, v in results.items() if v.get("status") == "success"}
}
with open(OUTPUT_DIR / "_report_pymu.json", "w") as f:
    json.dump(report, f, indent=2)

print(f"\n✅ Success : {report['success']}")
print(f"⏭️  Skipped : {report['skipped']}")
print(f"❌ Failed  : {len(failed)}")
if failed:
    print(f"Failed IDs : {failed}")

目標: 18 份


Processing with pymupdf:   6%|▌         | 1/18 [00:00<00:04,  3.64it/s]

✅ 2410.10516v3: 83803 chars, 25 pages, 5 images


Processing with pymupdf:  11%|█         | 2/18 [00:01<00:10,  1.49it/s]

✅ 2412.00651v1: 97226 chars, 21 pages, 89 images


...

✅ 2410.08642v2: 65910 chars, 12 pages, 10 images


Processing with pymupdf:  94%|█████████▍| 17/18 [00:28<00:02,  2.09s/it]

✅ 2411.04950v4: 173090 chars, 79 pages, 19 images


Processing with pymupdf: 100%|██████████| 18/18 [00:29<00:00,  1.62s/it]

✅ 2411.05000v2: 67898 chars, 22 pages, 1 images

✅ Success : 18
⏭️  Skipped : 0
❌ Failed  : 0
